# PLL lock range against the supply

Results of `pll_range.edge_search` (`sw/tests/pll_range.py`): per PLL supply the lowest (to 10 MHz) and highest (to 100 MHz) VCO frequency that locks and still holds the lock after `EDGE_HOLD` s, with the PLL supply power at the edges. Lock check only, no jitter.

In [19]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

PLL_DIR = Path.cwd().resolve().parents[2] / 'results' / 'pll'  # notebook lives in sw/tools/notebooks/
# Top edge: the rerun of 2026-11-06 (3.54 GHz at 0.50 V to >= 12.8 GHz at 0.90 V). None = newest run
EDGES_CSV = PLL_DIR / 'pll_edges_S5_20261106_110104.csv'
# Bottom edge from another run (its lower_* summary columns replace those of EDGES_CSV), here the bottom-only run
# after the top rerun. 'newest' = the newest run; None = the bottom of EDGES_CSV itself
BOTTOM_CSV = 'newest'


def newest_run():
    return sorted((p for p in PLL_DIR.glob('pll_edges_*.csv') if not p.stem.endswith('_summary')),
                  key=lambda p: p.stat().st_mtime)[-1]


def summary_of(csv):
    csv = Path(csv)
    return pd.read_csv(csv.with_name(csv.stem + '_summary.csv')).sort_values('vdd_set').reset_index(drop=True)


EDGES_CSV = Path(EDGES_CSV) if EDGES_CSV is not None else newest_run()
BOTTOM_CSV = newest_run() if BOTTOM_CSV == 'newest' else BOTTOM_CSV
if BOTTOM_CSV is not None and Path(BOTTOM_CSV) == EDGES_CSV:
    BOTTOM_CSV = None  # the bottom run has not been written yet: only the top
probes = pd.read_csv(EDGES_CSV)
summary = summary_of(EDGES_CSV)
RUN_TXT = EDGES_CSV.stem
OUT_CSV = EDGES_CSV  # the figures are saved next to this run
if BOTTOM_CSV is not None:
    BOTTOM_CSV = Path(BOTTOM_CSV)
    probes = pd.concat([probes[probes['edge'] != 'lower'], pd.read_csv(BOTTOM_CSV).query("edge == 'lower'")])
    lower_cols = [c for c in summary.columns if c.startswith('lower_')]
    summary = summary.drop(columns=lower_cols).merge(summary_of(BOTTOM_CSV)[['vdd_set'] + lower_cols],
                                                     on='vdd_set', how='outer')
    RUN_TXT = f'top {EDGES_CSV.stem}, bottom {BOTTOM_CSV.stem}'
    OUT_CSV = BOTTOM_CSV.with_name(BOTTOM_CSV.stem + '_combined.csv')

table = pd.DataFrame({
    'supply [V]': summary['vdd_set'],
    'measured [V]': summary['vdd_meas'].round(4),
    'lowest held [MHz]': (summary['lower_f'] / 1e6).round(1),
    'fails at [MHz]': (summary['lower_f_fail'] / 1e6).round(1),
    'P bottom [mW]': (summary['lower_p'] * 1e3).round(3),
    'highest held [GHz]': (summary['upper_f'] / 1e9).round(3),
    'fails at [GHz]': (summary['upper_f_fail'] / 1e9).round(3),
    'P top [mW]': (summary['upper_p'] * 1e3).round(3),
    'P top / f [mW/GHz]': (summary['upper_p'] * 1e3 / (summary['upper_f'] / 1e9)).round(3),
    'probes': summary['lower_n_probes'].fillna(0) + summary['upper_n_probes'].fillna(0),
})
print(f'{RUN_TXT}: {len(summary)} supplies, {len(probes)} tries')
print('fails at empty with a held edge: nothing failed up to the search limit (top: the 100 MHz reference, '
      'VCO 12.8 GHz); both empty: not searched or nothing held')
display(table)


def save(fig, name):
    """Interactive HTML and a static PNG (kaleido) next to OUT_CSV (<stem>_<name>), then show the figure."""
    base = OUT_CSV.with_name(f'{OUT_CSV.stem}_{name}')
    fig.write_html(base.with_suffix('.html'))
    try:
        fig.write_image(base.with_suffix('.png'), width=1100, height=fig.layout.height or 600, scale=2)
        print(f'saved {base}.html and .png')
    except Exception as e:  # kaleido >= 1.0 needs a Chrome/Chromium install (`plotly_get_chrome`)
        print(f'saved {base}.html; PNG export failed: {e}')
    fig.show()

top pll_edges_S5_20261106_110104, bottom pll_edges_S5_20261106_112507: 9 supplies, 292 tries
fails at empty with a held edge: nothing failed up to the search limit (top: the 100 MHz reference, VCO 12.8 GHz); both empty: not searched or nothing held


,supply [V],measured [V],lowest held [MHz],fails at [MHz],P bottom [mW],highest held [GHz],fails at [GHz],P top [mW],P top / f [mW/GHz],probes
0,0.50,0.5013,128.0,118.4,0.118,3.541,3.628,0.435,0.123,30
1,0.55,0.5514,108.8,100.6,0.129,5.427,5.494,0.728,0.134,23
2,0.60,0.6015,128.0,118.4,0.150,7.362,7.452,1.105,0.150,16
3,0.65,0.6516,128.0,118.4,0.165,9.282,9.339,1.584,0.171,12
4,0.70,0.7001,128.0,118.4,0.179,10.811,10.878,2.088,0.193,9
5,0.75,0.7502,144.9,139.3,0.197,11.919,11.992,2.590,0.217,9
6,0.80,0.8003,139.3,133.6,0.213,12.414,12.478,2.995,0.241,10
7,0.85,0.8504,128.0,118.4,0.228,12.736,12.800,3.390,0.266,9
8,0.90,0.9005,644.1,638.0,0.510,12.800,NaN,3.736,0.292,18


## Lock range per supply

One bar per supply from the lowest to the highest held VCO frequency (log axis); a single tick when only one edge was searched. A `≥` label: nothing failed up to the search limit, the real edge lies higher.

In [20]:
BAR_COLOR = '#2a6fdb'

fig = go.Figure()
for _, r in summary.iterrows():
    ends = [(edge, r[f'{edge}_f'] / 1e9, r[f'{edge}_f_fail'], r[f'{edge}_p'] * 1e3) for edge in ('lower', 'upper')
            if not pd.isna(r[f'{edge}_f'])]
    if not ends:
        continue
    text = [(f'{f * 1e3:.0f} MHz' if edge == 'lower' else ('≥ ' if pd.isna(fail) else '') + f'{f:.2f} GHz')
            for edge, f, fail, _ in ends]
    fig.add_trace(go.Scatter(
        x=[r['vdd_set']] * len(ends), y=[f for _, f, _, _ in ends], mode='lines+markers+text',
        line=dict(width=26, color=BAR_COLOR), opacity=0.85,
        marker=dict(symbol='line-ew', size=34, line=dict(width=3, color='black')),
        text=text, textposition=['bottom center' if edge == 'lower' else 'top center' for edge, *_ in ends],
        showlegend=False,
        customdata=[[p, fail / (1e6 if edge == 'lower' else 1e9)] for edge, _, fail, p in ends],
        hovertemplate=(f"supply {r['vdd_set']:.2f} V<br>" + 'VCO %{y:.3f} GHz<br>P %{customdata[0]:.3f} mW'
                       '<br>fails at %{customdata[1]:.3f} (MHz bottom / GHz top)<extra></extra>')))

searched = [e for e in ('lower', 'upper') if summary[f'{e}_n_probes'].notna().any()]
for _, r in summary.iterrows():
    if any(pd.isna(r[f'{e}_f']) for e in searched):
        fig.add_annotation(x=r['vdd_set'], y=0, yref='paper', text='no lock', showarrow=False, yshift=12)

fig.update_layout(title=f'Lock range against the PLL supply ({RUN_TXT})', height=600,
                  template='plotly_white', margin=dict(t=60))
fig.update_xaxes(title='PLL supply [V]', tickvals=summary['vdd_set'], ticktext=[f'{v:.2f}' for v in summary['vdd_set']],
                 range=[summary['vdd_set'].min() - 0.04, summary['vdd_set'].max() + 0.04])
fig.update_yaxes(title='VCO frequency [GHz]', type='log', range=[np.log10(0.03), np.log10(20)])
save(fig, 'range')

Resorting to unclean kill browser.


saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_edges_S5_20261106_112507_combined_range.html and .png


## Power at the edges

PLL supply power at the highest and lowest held frequency, and the power per GHz at the top.

In [21]:
from plotly.subplots import make_subplots

fig = make_subplots(rows=1, cols=2, subplot_titles=('Power at the edges', 'Power per GHz at the top'))
for edge, name, color in (('upper', 'top', '#d1495b'), ('lower', 'bottom', '#2a6fdb')):
    fig.add_trace(go.Scatter(x=summary['vdd_set'], y=summary[f'{edge}_p'] * 1e3, mode='lines+markers', name=name,
                             line=dict(color=color),
                             customdata=summary[f'{edge}_f'] / 1e6,
                             hovertemplate='%{x:.2f} V<br>%{y:.3f} mW at %{customdata:.0f} MHz<extra></extra>'),
                  row=1, col=1)
fig.add_trace(go.Scatter(x=summary['vdd_set'], y=summary['upper_p'] * 1e3 / (summary['upper_f'] / 1e9),
                         mode='lines+markers', name='top, mW/GHz', line=dict(color='#d1495b', dash='dot'),
                         hovertemplate='%{x:.2f} V<br>%{y:.3f} mW/GHz<extra></extra>'), row=1, col=2)
fig.update_xaxes(title='PLL supply [V]')
fig.update_yaxes(title='P [mW]', row=1, col=1)
fig.update_yaxes(title='P / f [mW/GHz]', rangemode='tozero', row=1, col=2)
fig.update_layout(height=450, template='plotly_white')
save(fig, 'power')

saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_edges_S5_20261106_112507_combined_power.html and .png


## Every probe

Check on the search: every probed frequency per supply, held or not. A failed probe inside a bar is a hole in the lock range that the bracketing assumed away.

In [22]:
def result_of(r):
    if r['held'] == 1:
        return 'held'
    if r.get('locked') == 1:
        return 'locked, lost again'
    return {'no_vco_setting': 'no VCO setting', 'error': 'error'}.get(r['status'], 'not locked')


probes['result'] = probes.apply(result_of, axis=1)
# a bottom probe has several tries (filters, |Kvco| picks): show the held one, else the last
probes = probes.sort_values('held', kind='stable').drop_duplicates(['vdd_set', 'edge', 'f_vco'], keep='last')
COLORS = {'held': '#2a9d4b', 'locked, lost again': '#f4a259', 'not locked': '#d1495b', 'no VCO setting': '#999999',
          'error': 'black'}
fig = go.Figure()
for result, color in COLORS.items():
    d = probes[probes['result'] == result]
    if d.empty:
        continue
    fig.add_trace(go.Scatter(
        x=d['vdd_set'] + np.where(d['edge'] == 'upper', 0.006, -0.006), y=d['f_vco'] / 1e9, mode='markers',
        name=result, marker=dict(color=color, size=9, symbol='circle'),
        customdata=np.stack([d['edge'], d['vco_pick'].fillna(''), d['pll_p'] * 1e3], axis=-1),
        hovertemplate='VCO %{y:.4f} GHz<br>%{customdata[0]} edge, pick %{customdata[1]}'
                      '<br>P %{customdata[2]:.3f} mW<extra></extra>'))
fig.update_layout(title='Every probe of the edge search (bottom search left, top search right of each supply)',
                  height=600, template='plotly_white')
fig.update_xaxes(title='PLL supply [V]', tickvals=summary['vdd_set'])
fig.update_yaxes(title='VCO frequency [GHz]', type='log')
save(fig, 'probes')

saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_edges_S5_20261106_112507_combined_probes.html and .png
